# Raw — Fetch OECD IP charges data via SDMX API

Fetches **"Charges for the use of intellectual property, n.i.e."** (service code `SH`)
from the OECD Balance of Payments Trade in Services dataset.

| | |
|---|---|
| **Source** | OECD SDMX REST API (public, no auth) |
| **Dataset** | `OECD.SDD.TPS, DSD_BOP@DF_TIS, v1.0` |
| **Coverage** | Canada vs. all counterpart countries · annual · balance · CAD |
| **Output** | `data/raw/oecd/raw_oecd__ip_charges.json` |

See `docs/oecd-ip-charges.md` for full documentation.

## 1. Setup

In [5]:
import json
import requests
from pathlib import Path
import sys

project_root = Path.cwd().parent.parent.parent
sys.path.insert(0, str(project_root))

from utils.paths import RAW_OECD_DIR

RAW_OECD_DIR.mkdir(parents=True, exist_ok=True)

## 2. API configuration

The dimension key `CAN..SH.B..A.XDC.` breaks down as:
- `CAN` — Canada (REF_AREA)
- `.` — all counterpart countries (COUNTERPART_AREA)
- `SH` — charges for use of IP (MEASURE)
- `B` — balance, i.e. credits minus debits (ACCOUNTING_ENTRY)
- `.` — all (FS_ENTRY; only Total is present)
- `A` — annual (FREQ)
- `XDC` — national/domestic currency, i.e. CAD for Canada (UNIT_MEASURE)
- `.` — all (ADJUSTMENT; only Not adjusted is present)

In [6]:
OECD_API_BASE   = 'https://sdmx.oecd.org/public/rest/data'
DATASET_AGENCY  = 'OECD.SDD.TPS'
DATASET_ID      = 'DSD_BOP@DF_TIS'
DATASET_VERSION = '1.0'

DIMENSION_KEY = 'CAN..SH.B..A.XDC.'

# No startPeriod filter — fetch the full series from the beginning.
# Coverage varies by counterpart: world totals go back the furthest,
# the Canada-US bilateral series starts ~1988, most others from ~2020.

OUTPUT_FILENAME = 'raw_oecd__ip_charges.json'

## 3. Fetch from OECD API

In [7]:
url = f'{OECD_API_BASE}/{DATASET_AGENCY},{DATASET_ID},{DATASET_VERSION}/{DIMENSION_KEY}'
params = {
    # dimensionAtObservation=AllDimensions puts TIME_PERIOD in the observation key,
    # which gives us a flat structure that's easier to parse than the default time-series layout
    'dimensionAtObservation': 'AllDimensions',
}
# Request SDMX-JSON format instead of the default SDMX-ML (XML)
headers = {
    'Accept': 'application/vnd.sdmx.data+json',
}

print(f'URL:    {url}')
print(f'Params: {params}')
print()

response = requests.get(url, params=params, headers=headers, timeout=60)
response.raise_for_status()

print(f'Status:       HTTP {response.status_code} OK')
print(f'Content-Type: {response.headers.get("Content-Type", "unknown")}')

data = response.json()

URL:    https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_BOP@DF_TIS,1.0/CAN..SH.B..A.XDC.
Params: {'dimensionAtObservation': 'AllDimensions'}

Status:       HTTP 200 OK
Content-Type: application/vnd.sdmx.data+json; version=2; charset=utf-8


In [8]:
output_path = RAW_OECD_DIR / OUTPUT_FILENAME
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f'Raw data saved to: {output_path}')

Raw data saved to: C:\Users\GuidaK\OneDrive - ISED-ISDE\Documents\Work\or_country_profiles_dashboard\data\raw\oecd\raw_oecd__ip_charges.json


## 4. Validate response

Quick sanity check — confirm we got data and see what years/countries are covered.

In [9]:
d        = data['data']
dataset  = d['dataSets'][0]
struct   = d['structures'][0]

print(f'Total observations: {len(dataset["observations"])}')
print()

for dim in struct['dimensions']['observation']:
    if dim['id'] == 'TIME_PERIOD':
        years = sorted(v['id'] for v in dim['values'])
        print(f'Years covered ({len(years)}): {years[:5]} ... {years[-5:]}')
    if dim['id'] == 'COUNTERPART_AREA':
        countries = [v['name'] for v in dim['values']]
        preview   = countries[:8]
        print(f'Counterpart areas ({len(countries)}): {preview}{" ..." if len(countries) > 8 else ""}')
    if dim['id'] == 'MEASURE':
        measures = [(v['id'], v['name']) for v in dim['values']]
        print(f'Measures in response: {measures}')
        print('  → staging will filter to SH only')

Total observations: 1323

Counterpart areas (92): ['Belgium', 'France', 'Italy', 'Switzerland', 'European Union (28 countries)', 'America', 'Europe', 'Africa'] ...
Measures in response: [('SH', 'Charges for the use of intellectual property n.i.e.'), ('S', 'Services'), ('CA', 'Current account')]
  → staging will filter to SH only
Years covered (56): ['1969', '1970', '1971', '1972', '1973'] ... ['2020', '2021', '2022', '2023', '2024']
